In [3]:
import os
import numpy as np
import pandas as pd
import boto3
import time
import helper
import sys
cwd = os.getcwd()
print(cwd)

/home/sagemaker-user/CAPE_PERFORMANCE


#### Load model baseline

In [4]:
# read data from baseline
bucket_name = 'pr-home-datascience'
prefix = 'Projects/Underwriting/UnconstraintModels/HO_UNCS_V0140_2023/production/m14_modeling_wo_cape_research/'
# s3://pr-home-datascience/Projects/Underwriting/UnconstraintModels/HO_UNCS_V0140_2023/production/m14_modeling_wo_cape_research/m140_sample.pq
s3 = boto3.client('s3')

response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

if 'Contents' in response:
    parquet_files = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.parquet')]
    print("Found parquet files:")
    for f in parquet_files:
        print(f)
else:
    print("No files found under that prefix.")


path = f"s3://{bucket_name}/{prefix}m140_sample.pq"
print(f"Reading {prefix}...")
df = pd.read_parquet(path)
dfs = []
dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
df_baseline = df_all
print("Merged total rows:", len(df_all))


Found parquet files:
Reading Projects/Underwriting/UnconstraintModels/HO_UNCS_V0140_2023/production/m14_modeling_wo_cape_research/...
Merged total rows: 10691064


Check the data structures

In [5]:
keywords = 'dt'

key_columns = [col for col in df_baseline.columns if keywords in col.lower()]
print('There are ', len(key_columns), ' variables contains ' + keywords + '.')
print('They are ', key_columns)
df_baseline['year'].head()

There are  0  variables contains dt.
They are  []


0    2017
1    2017
2    2017
3    2017
4    2017
Name: year, dtype: int64

In [6]:
print(df_baseline[df_baseline['akey_or_polnum']=='BH_000000000000018'] )

             qpid  year state    zip5    zip4 tv      akey_or_polnum  book  \
0        65486532  2017    MA  1960.0  5731.0  T  BH_000000000000018  LXNX   
647945   65486532  2019    MA  1960.0  5731.0  T  BH_000000000000018  LXNX   
647946   65486532  2020    MA  1960.0  5731.0  T  BH_000000000000018  LXNX   
647947   65486532  2021    MA  1960.0  5731.0  T  BH_000000000000018  LXNX   
647948   65486532  2022    MA  1960.0  5731.0  T  BH_000000000000018  LXNX   
5544295  65486532  2018    MA  1960.0  5731.0  T  BH_000000000000018  LXNX   
5544296  65486532  2023    MA  1960.0  5731.0  T  BH_000000000000018  LXNX   

                window   ee  ...  p5_wate_cnt1  p6_wwat_cnt1  p7_weat_cnt1  \
0        201107-201706  6.0  ...           0.0           0.0           0.0   
647945   201807-201906  1.0  ...           0.0           0.0           0.0   
647946   201907-202006  1.0  ...           0.0           0.0           0.0   
647947   202007-202106  1.0  ...           0.0           0.0   

In [7]:
print(f'The length of model baseline is {len(df_baseline)}')
print(f'The length of model baseline variables is {len(df_baseline.columns)}')
print(f'The variables of model baseline is {df_baseline.columns}')



The length of model baseline is 10691064
The length of model baseline variables is 119
The variables of model baseline is Index(['qpid', 'year', 'state', 'zip5', 'zip4', 'tv', 'akey_or_polnum', 'book',
       'window', 'ee',
       ...
       'p5_wate_cnt1', 'p6_wwat_cnt1', 'p7_weat_cnt1', 'p8_acat_cnt1',
       'p9_awat_cnt1', 'legacy_mask', 'tv_c', 'flag', 'p0_alxc_target',
       'p0_alxc_wo_cape_pred'],
      dtype='object', length=119)


#### Load Cape and model data

In [11]:
model_prefix = 'Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/'
bucket_name = 'pr-home-datascience'
months = ['may', 'july']
versions = ['v4', 'v5']
splits = ['train', 'val']

out_dir = 's3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/baseline_match/'
v = versions[0]
split = splits[0]
model_file = f"model_{split}_{v}.csv"
cape_file = f"cape_{split}_{v}.csv"
df_model  = helper.read_s3(bucket_name, model_prefix, model_file)
df_cape = helper.read_s3(bucket_name, model_prefix, cape_file)

Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/model_train_v4.csv


Load file successfully, file length is  940672
Now the total rows are  940672
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/cape_train_v4.csv
Load file successfully, file length is  940672
Now the total rows are  940672


#### Match the model (cape) with baseline
Since cape and model are already matched, we only need to match model and baseline. Then use the index to match the cape data from model data.\
For the baseline, since it only contain 'pol_num' and 'year' information, we will use them when matching with model.

In [12]:
def match_model_to_baseline(prefix, df_baseline, split, v, save_base=None):
    """
    df_baseline: DataFrame with at least ['pol_num','cur_term_eff_dt','ncat','ee']
    split: 'train' | 'val'
    v: e.g., 'v4', 'v5'
    m: month token used in filenames, e.g., 'May', 'July', or '2025-05'
    save_base: where to save outputs (S3 or local). If None, uses model_base_path.
    """

    # ----- Load model file -----
    model_file = f"model_{split}_{v}.csv"
    cape_file = f"cape_{split}_{v}.csv"
    bucket_name =  'pr-home-datascience'
    print('The prefix is ', prefix)
    df_model  = helper.read_s3(bucket_name, prefix, model_file)
    df_cape = helper.read_s3(bucket_name, prefix, cape_file)
    # return  df_model, df_cape, df_baseline,


    df_model['year'] = df_model['year'].astype('Int64')
    keys_to_match = ['pol_num', 'year']
    df_model = df_model.drop_duplicates(subset=keys_to_match, keep='first')
    df_cape['year'] = df_model['year']

    # Rename baseline key
    df_base_renamed = df_baseline.rename(columns={'akey_or_polnum': 'pol_num'})
    df_base_renamed['year'] = df_base_renamed['year'].astype('Int64')

    # Define the common merge keys
    
    model_keys = df_model[keys_to_match].drop_duplicates()
    base_keys = df_base_renamed[keys_to_match].drop_duplicates()

    # Find the final set of keys that exist in BOTH
    common_keys = pd.merge(
        model_keys,
        base_keys,
        on=keys_to_match,
        how='inner'
    )
    print(f"Found {len(common_keys)} unique common (year, pol_num) keys.")


    # --- Step 3: Filter df_model and df_baseline ---
    df_matched_model = pd.merge(
        df_model,
        common_keys,
        on=keys_to_match,
        how='inner'
    )
    df_matched_base = pd.merge(
        df_base_renamed,
        common_keys,
        on=keys_to_match,
        how='inner'
    )
    df_matched_cape = pd.merge(
        df_cape,
        common_keys,
        on=keys_to_match,
        how='inner'
    )

    print("Step 5: Sorting all three DataFrames for perfect alignment...")

    # 1. Sort model and base (this is easy, they have the keys)
    df_matched_model = df_matched_model.sort_values(by=keys_to_match)
    df_matched_base = df_matched_base.sort_values(by=keys_to_match)
    df_matched_cape = df_matched_cape.sort_values(by=keys_to_match)


    # --- Step 6: Final Reset for Identical DataFrames ---
    print("Step 6: Resetting index...")
    df_matched_model = df_matched_model.reset_index(drop=True)
    df_matched_base = df_matched_base.reset_index(drop=True)
    df_matched_cape = df_matched_cape.reset_index(drop=True)

    # 4. (Optional) Remove the temporary keys from df_cape
    df_matched_cape = df_matched_cape.drop(columns='year')

    print("--- Alignment Complete ---")
    print(f"Final shape df_matched_model: {df_matched_model.shape}")
    print(f"Final shape df_matched_cape: {df_matched_cape.shape}")
    print(f"Final shape df_matched_base: {df_matched_base.shape}")

    # Show a few side-by-side to confirm
    # print("\n[Sanity check — first 10]")
    # print(base_common[["year","pol_num_clean"]].head(10))
    # print(df_matched_model[["cur_term_eff_dt","year","pol_num_clean"]].head(10))
    # print(cape_common[["cape_run_dt","pol_num"]].head(10))

    baseline_out   = f"{save_base}matchedbase_baseline_{split}_{v}.csv"
    model_out   = f"{save_base}matchedbase_model_{split}_{v}.csv"
    cape_out   = f"{save_base}matchedbase_cape_{split}_{v}.csv"

    df_matched_model.to_csv(model_out, index=False)
    df_matched_cape.to_csv(cape_out, index=False)
    df_matched_base.to_csv(baseline_out, index=False)

    return  df_matched_model, df_matched_cape, df_matched_base,




In [9]:
model_prefix = 'Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/'
months = ['may', 'july']
versions = ['v4', 'v5']
splits = ['train', 'val']
bucket_name = 'pr-home-datascience'

out_dir = 's3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/base_model_cape_aligned_train_val/'


for v in versions:
    for split in splits:
        # baseline_common, model_common, cape_common = match_model_to_baseline(model_prefix, df_baseline, split, v, save_base=out_dir)
        _ = match_model_to_baseline(model_prefix, df_baseline, split, v, save_base=out_dir)

    

The prefix is  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/model_train_v4.csv


Load file successfully, file length is  940672
Now the total rows are  940672
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/cape_train_v4.csv
Load file successfully, file length is  940672
Now the total rows are  940672
Found 655887 unique common (year, pol_num) keys.
Step 5: Sorting all three DataFrames for perfect alignment...
Step 6: Resetting index...
--- Alignment Complete ---
Final shape df_matched_model: (655887, 15)
Final shape df_matched_cape: (655887, 103)
Final shape df_matched_base: (655887, 119)
The prefix is  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/model_val_v4.csv
Load file successfully, file length is  400586
Now the total rows are  400586
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/train_val/cape

#### Sanity check

In [11]:
print('Before the matching')
print(f'df_model has shape of {df_model.shape}')
print(f'df_cape has shape of {df_cape.shape}')
print(f'df_baseline has shape of {df_baseline.shape}')

Before the matching
df_model has shape of (655887, 15)
df_cape has shape of (655887, 103)
df_baseline has shape of (655887, 119)


The following is May examples, which we use 'effective_date' in Cape and 'cur_term_eff_dt' in model. Those two could be the same.

In [13]:
print(
    df_model.loc[
        df_model['pol_num'] == 'NYH00002001317',  # This is your row filter
        ['year', 'cur_term_eff_dt', 'pol_num']   # This is your column list
    ]
)

print(
    df_cape.loc[
        df_cape['pol_num'] == 'NYH00002001317',  # This is your row filter
        [ 'cape_run_dt', 'pol_num']   # This is your column list
    ]
)

        year cur_term_eff_dt         pol_num
428696  2020      2020-12-02  NYH00002001317
428697  2021      2021-12-02  NYH00002001317
428698  2022      2022-12-02  NYH00002001317
       cape_run_dt         pol_num
428696  2020-12-02  NYH00002001317
428697  2021-12-02  NYH00002001317
428698  2022-12-02  NYH00002001317


The following examples show both May and July records.

In [12]:
print(df_baseline[['year', 'pol_num']].iloc[0:20] )
print(df_model[['year', 'cur_term_eff_dt', 'pol_num']].iloc[0:20] )
print(df_cape[['cape_run_dt', 'pol_num']].iloc[0:20] )

    year         pol_num
0   2014  BHD00001001024
1   2015  BHD00001001024
2   2016  BHD00001001024
3   2017  BHD00001001024
4   2018  BHD00001001024
5   2019  BHD00001001024
6   2020  BHD00001001024
7   2021  BHD00001001024
8   2014  BHD00001001041
9   2015  BHD00001001041
10  2016  BHD00001001041
11  2017  BHD00001001041
12  2018  BHD00001001041
13  2019  BHD00001001041
14  2020  BHD00001001041
15  2021  BHD00001001041
16  2022  BHD00001001041
17  2020  BHD00001001042
18  2021  BHD00001001042
19  2022  BHD00001001042
    year cur_term_eff_dt         pol_num
0   2014      2014-01-03  BHD00001001024
1   2015      2015-01-03  BHD00001001024
2   2016      2016-01-03  BHD00001001024
3   2017      2017-01-03  BHD00001001024
4   2018      2018-01-03  BHD00001001024
5   2019      2019-01-03  BHD00001001024
6   2020      2020-01-03  BHD00001001024
7   2021      2021-01-03  BHD00001001024
8   2014      2014-03-30  BHD00001001041
9   2015      2015-03-30  BHD00001001041
10  2016      2016-03-30

Some examples from July that Cape 'cape_run_dt' is earlier than model 'cur_term_eff_dt'.

In [94]:
model_compare = df_matched_model[['cur_term_eff_dt', 'pol_num']].rename(
    columns={'cur_term_eff_dt': 'dt'}
)
cape_compare = df_matched_cape[['cape_run_dt', 'pol_num']].rename(
    columns={'cape_run_dt': 'dt'}
)
are_they_identical = model_compare.equals(cape_compare)

print(f"\nAre the selected columns identically same? {are_they_identical}\n")

if not are_they_identical:
    print("Finding differences...")
    
    # 3. Perform an element-wise comparison.
    # We must reset the index on both sides for the comparison operator ( != )
    # to compare the *values* row-by-row, not the index labels.
    comparison = model_compare.reset_index(drop=True) != cape_compare.reset_index(drop=True)
    
    # 4. Find rows (by location) that have *any* difference
    different_rows_mask = comparison.any(axis=1)
    
    if different_rows_mask.any():
        # 5. Get the *original index labels* for those different rows.
        # We use the mask (which is 0-based) to select from the
        # actual index labels of the 'model_compare' DataFrame.
        different_indices = model_compare.index[different_rows_mask]
        
        print(f"Found differences at the following {len(different_indices)} index label(s):")
        print(list(different_indices))
        
        print("\n--- Data at different indices ---")
        
        print("\nModel Data:")
        # Use .loc[] to select by the original index labels
        print(df_matched_model.loc[different_indices[0:10], ['cur_term_eff_dt', 'pol_num']])
        
        print("\nCape Data:")
        # Use .loc[] to select by the original index labels
        print(df_matched_cape.loc[different_indices[0:10], ['cape_run_dt', 'pol_num']])
    else:
        print("No row-by-row differences found (this is unexpected if .equals() was False).")
else:
    print("All rows in the selected columns are identically the same.")


Are the selected columns identically same? False

Finding differences...
Found differences at the following 454116 index label(s):
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 62, 63, 64, 65, 66, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 113, 114, 115, 116, 117, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 

In [22]:
baseline_common.columns
print(baseline_common.iloc[0:20])

             qpid  year state    zip5    zip4 tv  akey_or_polnum    book  \
4535915  65164516  2014    MA  2740.0  1228.0  T  BHD00001001024  Legacy   
4535916  65164516  2015    MA  2740.0  1228.0  T  BHD00001001024  Legacy   
4535917  65164516  2017    MA  2740.0  1228.0  T  BHD00001001024  Legacy   
4535918  65164516  2018    MA  2740.0  1228.0  T  BHD00001001024  Legacy   
4535919  65164516  2020    MA  2740.0  1228.0  T  BHD00001001024  Legacy   
4535920  65164516  2021    MA  2740.0  1228.0  T  BHD00001001024  Legacy   
4535949  65178795  2014    MA  2745.0  5260.0  T  BHD00001001041  Legacy   
4535950  65178795  2016    MA  2745.0  5260.0  T  BHD00001001041  Legacy   
4535951  65178795  2017    MA  2745.0  5260.0  T  BHD00001001041  Legacy   
4535952  65178795  2018    MA  2745.0  5260.0  T  BHD00001001041  Legacy   
4535953  65178795  2019    MA  2745.0  5260.0  T  BHD00001001041  Legacy   
4535954  65178795  2020    MA  2745.0  5260.0  T  BHD00001001041  Legacy   
4535955  651

#### Check the Cape & model data no later than 2023/6
Since baseline is all early than June 2023, we want to see how many records for Cape and model before this time. \
Examine the original amount of data is similar with the results after matching. The answer is similar.

In [ ]:
df_model["cur_term_eff_dt"] = pd.to_datetime(df_model["cur_term_eff_dt"], errors="coerce")

mask = (df_model["cur_term_eff_dt"].dt.year < 2023) | \
       ((df_model["cur_term_eff_dt"].dt.year == 2023) & (df_model["cur_term_eff_dt"].dt.month <= 6))
df_leq_jun2023_v2 = df_model[mask].copy()

In [ ]:

print(f'The training dataset, the original model_cape match samples are {len(df_model)}')
print(f'The samples no late than 2023-6 are {len(df_leq_jun2023_v2)}')

The training dataset, the original model_cape match samples are 940672
The samples no late than 2023-6 are 679620


In [ ]:
print(model_only['pol_num'].unique()[0:20])
print(baseline_only['pol_num'].unique()[0:20])




['BHD00001001042' 'BHD00001001104' 'BHD00001001138' 'BHD00001001164'
 'BHD00001001196' 'BHD00001001241' 'BHD00001001320' 'BHD00001001348'
 'BHD00001001363' 'BHD00001001368' 'BHD00001001442' 'BHD00001001542'
 'BHD00001001580' 'BHD00001001662' 'BHD00001001841' 'BHD00001001882'
 'BHD00001001893' 'BHD00001001952' 'BHD00001002014' 'BHD00001002026']
['BHH00001001064' 'BHH00001001117' 'BHH00001001126' 'BHH00001001129'
 'BHH00001001136' 'BHH00001001138' 'BHH00001001145' 'BHH00001001158'
 'BHH00001001164' 'BHH00001001166' 'BHH00001001173' 'BHH00001001181'
 'BHH00001001201' 'BHH00001001208' 'BHH00001001227' 'BHH00001001247'
 'BHH00001001257' 'BHH00001001287' 'BHH00001001289' 'BHH00001001310']


In [ ]:
print(df_model["cur_term_eff_dt"][0:10])


print(df_baseline.query("pol_num == 'BHD00001001368'"))

print(model_only.query("pol_num == 'BHH00001001064'"))


0    2020-04-26
1    2021-04-26
2    2022-04-26
3    2023-04-26
4    2024-04-26
5    2025-04-26
6    2020-11-15
7    2021-11-15
8    2022-11-15
9    2023-11-15
Name: cur_term_eff_dt, dtype: object
Empty DataFrame
Columns: [pol_num, co_cd, ho_pol_form, cur_term_eff_dt, cur_term_xptn_dt, pol_state_eff_dt, row_xptn_dt, orgl_pol_eff_dt, agcy_cd, agcy_name, org_sub_channel_name, auto_pol_num, comm_grp, cmpnn_auto_pol_flg, quote_num, maxloss_dt, minloss_dt, sample, form, address, city, county, match_source, tv, pol_seq_num, new_or_rnwl_flg, year, pif_20251001, current_noic, prior_noic, p1othr_cnt, p2fire_cnt, p3liab_cnt, p4thft_cnt, p5wate_cnt, p6wwat_cnt, p7weat_cnt, p8ccat_cnt, ncat_cnt, highpoint, zip, zip4, pol_pk, qpid, ins_scor, cova, covb, covc, covd, cove, full_term_prm, ee, eprm, tot_loss_inc, p1othr, p2fire, p3liab, p4thft, p5wate, p6wwat, p7weat, p8ccat, ncat, pa_latitude, pa_longitude, rundt, state, zip5, akey_or_polnum, book, window, weight0, weight1, p0_alxc_cnt, p1_othr_cnt, p